In [ ]:
# Step 1: Clean uninstall
!pip uninstall -y torch torchvision torchaudio

# Step 2: Install compatible CPU version (IMPORTANT)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

# Step 3: Install your chatbot library
!pip install sentence-transformers

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully")

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
df = pd.read_csv("university_queries_test_1k.csv")

question_col = df.columns[0]
answer_col = df.columns[1]

df[question_col] = df[question_col].astype(str)
df[answer_col] = df[answer_col].astype(str)

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

df[question_col] = df[question_col].apply(clean_text)

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
question_embeddings = model.encode(df[question_col].tolist())

In [ ]:
def rule_based_response(user_input):
    text = user_input.lower()

    if "admission" in text or "eligibility" in text:
        return "Admission requires 10th & 12th pass with minimum 50% marks."

    if "fees" in text:
        return "Fees can be paid online or at the accounts office."

    if "hostel" in text:
        return "Hostel facilities include WiFi, security, and mess."

    if "attendance" in text:
        return "Minimum 75% attendance is required."

    if "scholarship" in text:
        return "Scholarships are available based on merit and category."

    return None

In [ ]:
def retrieval_response(user_input):
    user_clean = clean_text(user_input)
    user_embedding = model.encode([user_clean])

    similarity = cosine_similarity(user_embedding, question_embeddings)
    idx = np.argmax(similarity)

    # Confidence check
    if similarity[0][idx] < 0.4:
        return "Sorry, I didn't understand your question clearly."

    return df.iloc[idx][answer_col]

In [ ]:
def chatbot(user_input):
    rule = rule_based_response(user_input)

    if rule:
        return rule

    return retrieval_response(user_input)

In [ ]:
print("Chatbot is ready! Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Chatbot: Goodbye!")
        break

    response = chatbot(user_input)
    print("Chatbot:", response)

In [ ]:
# =========================
# Evaluation Section
# =========================

correct = 0
total = 100   # number of test samples

for i in range(total):
    question = df.iloc[i][question_col]
    actual_answer = df.iloc[i][answer_col]

    predicted_answer = retrieval_response(question)

    if predicted_answer == actual_answer:
        correct += 1

accuracy = correct / total

print("Model Accuracy:", accuracy)

In [ ]:
import matplotlib.pyplot as plt

methods = ["Your Chatbot"]
accuracies = [accuracy]

plt.figure()
plt.bar(methods, accuracies)

plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.title("Chatbot Performance")

plt.show()

In [ ]:
scores = []

for i in range(50):
    question = df.iloc[i][question_col]

    user_embedding = model.encode([question])
    similarity = cosine_similarity(user_embedding, question_embeddings)

    scores.append(np.max(similarity))

avg_similarity = np.mean(scores)

print("Average Similarity Score:", avg_similarity)